In [ ]:
import einops
from tqdm.notebook import tqdm
import torch
from torch import nn


In [2]:
# using GPU if available
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

patch_size = 16         # Patch size (P) = 16
latent_size = 768       # Latent vector (D). ViT-Base uses 768
n_channels = 3          # Number of channels for input images
num_heads = 12          # ViT-Base uses 12 heads
num_encoders = 12       # ViT-Base uses 12 encoder layers
dropout = 0.1           # Dropout = 0.1 is used with ViT-Base & ImageNet-21k
num_classes = 10        # Number of classes in CIFAR10 dataset
size = 224              # Size used for training = 224

epochs = 10             # Number of epochs
base_lr = 10e-3         # Base LR
weight_decay = 0.03     # Weight decay for ViT-Base (on ImageNet-21k)
batch_size = 1

cpu


Implementation of input layer projection

In [3]:
class InputEmbedding(nn.Module):
    def __init__(self, patch_size=patch_size,n_channels=n_channels, device=device, latent_size=latent_size, batch_size= batch_size):
        super(InputEmbedding,self).__init__()
        self.patch_size = patch_size
        self.latent_size = latent_size
        self.n_channels = n_channels
        self.device = device
        self.batch_size = batch_size
        self.input_size = self.patch_size * self.patch_size * self.n_channels
        # Linear Projection
        self.linearProjection = nn.Linear(self.input_size, self.latent_size)
        # Class token
        self.class_token = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)
        #Postional embedding
        self.pos_embedding = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)
    def forward(self, input_data):
        input_data = input_data.to(self.device)

        #Patchify input image
        patches = einops.rearrange(
            input_data, 'b c (h h1) (w w1) -> b (h w) (h1 w1 c)', h1=self.patch_size, w1=self.patch_size)
        
        # print(input_data.size())
        # print(patches.size())

        linear_projection = self.linearProjection(patches)
        b, n , _ = linear_projection.size()

        linear_projection = torch.cat((self.class_token, linear_projection), dim=1)
        pos_emb = einops.repeat(self.pos_embedding, 'b 1 d -> b m d', m=n+1)
        linear_projection += pos_emb
        return linear_projection

Encoder Block

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, latent_size=latent_size, num_heads= num_heads, device = device, dropout=dropout):
        super(EncoderBlock, self).__init__()
        self.latent_size=latent_size
        self.num_heads=num_heads
        self.device=device
        self.dropout=dropout

        # Normalization Layer 
        self.norm = nn.LayerNorm(self.latent_size)

        self.multihead = nn.MultiheadAttention(
            self.latent_size, self.num_heads, dropout=self.dropout
        )    
        self.enc_MLP = nn.Sequential(
            nn.Linear(latent_size, latent_size*4),
            nn.GELU(),
            nn.Dropout(self.dropout),
            nn.Linear(latent_size*4, latent_size),
            nn.Dropout(self.dropout)
        )
    def forward(self, embedded_patches):
        firstnorm_out = self.norm(embedded_patches)
        attention_out = self.multihead(firstnorm_out,firstnorm_out,firstnorm_out)[0]

        # first residual connection 
        first_added = attention_out + embedded_patches

        secodnorm_out = self.norm(first_added)
        ff_out = self.enc_MLP(secodnorm_out)
        out = ff_out + first_added
        print(out.size())
        return ff_out + first_added



Test

In [5]:
test_input = torch.randn((1, 3, 224, 224))
input_embedding = InputEmbedding()
embed_test = input_embedding(test_input)

In [6]:
test_encoder = EncoderBlock()
test_encoder(embed_test)

torch.Size([1, 197, 768])
torch.Size([1, 197, 768])


tensor([[[-1.3388,  0.4120,  0.6111,  ..., -0.7334,  1.5195, -4.3896],
         [-1.3961,  0.2345,  1.1435,  ...,  0.9695, -0.3213, -1.9469],
         [-1.5482,  0.2538,  1.6029,  ...,  0.4742,  0.1693, -1.5955],
         ...,
         [-1.3113, -0.4460,  1.5187,  ...,  1.5837,  0.7679, -1.0373],
         [-0.6912,  0.2502,  1.1468,  ..., -0.1043,  1.6058, -0.8231],
         [-2.2139, -1.1165,  1.1433,  ..., -0.5628,  0.4745, -0.6780]]],
       grad_fn=<AddBackward0>)

In [7]:
class Vit(nn.Module):
    def __init__(self, num_encoders=num_encoders, latent_size=latent_size, device=device, num_classes=num_classes, dropout=dropout):
        super(Vit, self).__init__()
        self.num_encoders=num_encoders
        self.latent_size=latent_size
        self.num_classes=num_classes
        self.dropout=dropout

        self.embedding = InputEmbedding()

        # create the stack of encoders 
        self.encStack = nn.ModuleList([EncoderBlock() for i in range(self.num_encoders)])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(self.latent_size),
            nn.Linear(self.latent_size,self.latent_size),
            nn.Linear(self.latent_size, self.num_classes)
        )
    def forward(self, test_input):
        enc_output = self.embedding(test_input)
        for enc_layer in self.encStack :
            enc_output = enc_layer(enc_output)
        cls_token_embed = enc_output[:,0]

        return self.mlp_head(cls_token_embed) 

Test 

In [8]:
model = Vit().to(device)

vit_output = model(test_input.to(device))
print(vit_output)
print(vit_output.size())

torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
torch.Size([1, 197, 768])
tensor([[-0.1834, -0.4051,  0.2084, -0.7815, -0.3208, -0.1947, -0.3570,  0.1139,
         -0.1521,  0.2184]], grad_fn=<AddmmBackward0>)
torch.Size([1, 10])
